# Two-Layer Wire Coating: Axisymmetric Flow of Power-Law Melts in a Die Land

*A self-contained teaching notebook for senior and graduate students in chemical engineering*

---

**What this notebook does.** It predicts the fully developed, laminar flow of **two concentric polymer melt layers** in the die land of a wire-coating die. The wire moves along the axis at the line speed, and the die wall is stationary. Each melt is a **power-law fluid**. Given the wire and die dimensions, the line speed, the melt properties, and either the layer flow rates or the final coating thicknesses, the notebook calculates:

* the **pressure gradient** (and pressure drop) the die must supply,
* the **position of the interface** between the two layers inside the die,
* the **velocity, shear-rate and viscosity profiles** across the gap,
* the **shear stress on the wire and the die wall**, and the **force needed to pull the wire** through the land.

**How to use it.**
1. Run the cells **from top to bottom** (in Jupyter: *Run → Run All Cells*, or `Shift + Enter` on each cell). Later cells use functions defined in earlier ones.
2. Read the text cells: they explain the physics, the mathematics and the code in that order.
3. Cells marked ✏️ are meant for you to edit and re-run.
4. Short "check your understanding" questions have hidden answers. Click **▸ Answer** to reveal one.

**Requirements:** Python 3 with `numpy`, `scipy` and `matplotlib`, all included in standard scientific Python distributions such as Anaconda.

## Learning objectives

After working through this notebook you should be able to:

1. **Describe** the wire-coating process and explain why the die land is modelled as fully developed annular flow with a moving inner wall.
2. **Derive** the shear-stress distribution $\tau_{rz}(r) = -Gr/2 + C/r$ from a shell momentum balance, and explain why one pair of constants $(G, C)$ holds in *every* layer.
3. **Apply** the power-law constitutive equation, with correct signs, to obtain velocity profiles and layer flow rates.
4. **Formulate** a multilayer flow problem as a set of nonlinear equations: count the unknowns and equations, and state the boundary and interface conditions.
5. **Explain** how the problem is solved numerically: quadrature, the special treatment of the zero-stress radius, nonlinear least squares, and continuation.
6. **Verify** a numerical model against analytical limiting cases, and judge whether the model's assumptions hold for a given operating point.
7. **Use** the model to answer process questions: How much pressure does the die need? How hard must the wire be pulled? Where is the interface? When does backflow occur?

**Prerequisites:** undergraduate transport phenomena (shell balances, Newtonian annular flow as in Bird, Stewart and Lightfoot, Chapter 2), the idea of a shear-thinning fluid, and basic Python.

## Contents

1. [The process: wire coating](#part1)
2. [Model formulation](#part2): assumptions, momentum balance, power-law fluid, boundary and interface conditions, mass balance
3. [Numerical method](#part3): quadrature, nonlinear equations, solution strategy
4. [The code, cell by cell](#part4)
5. [Verification against exact solutions](#part5)
6. [Worked examples](#part6)
7. [Your own calculation ✏️](#part7)
8. [Exercises](#part8)
9. [Summary and references](#part9)

Nomenclature is collected in Section 1.3.

<a id="part1"></a>
# 1. The process: wire coating

### 1.1 What happens on a wire-coating line

Electrical wire and cable are insulated by **extrusion coating**. A bare conductor (copper or aluminium) is unwound, often preheated, and pulled at constant speed through a **crosshead die** mounted on one or more extruders. Molten polymer is delivered to the die, flows around the wire, and leaves the die as a coating. A water trough then cools the coating, a capstan pulls the line, and the finished wire is wound up. Line speeds range from well under 1 m/s for heavy cables to tens of m/s for thin building wire.

In a **pressure-type die**, the melt meets the wire inside the die and the two travel together through the **die land**, a short straight annular passage just before the exit. The melt is driven partly by the die pressure and partly by the moving wire, which drags it along (*drag flow*). When two extruders feed one crosshead, two layers are applied at once (**coextrusion**). Typical pairs are a primary insulation under a protective jacket, or a foamed layer under a solid skin.

### 1.2 What we model

We model the **die land**: the straight annulus between the wire (radius $R_w$) and the die wall (radius $R_d$). Far enough from its entrance, the flow there is **fully developed**: the velocity depends on the radius only, not on the axial position. The schematic below shows the geometry. Run the cell to draw it.

In [ ]:
# Draw a schematic of the die land (illustration only; no calculations here).
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon, FancyArrowPatch

def draw_wire_coating_schematic():
    """Longitudinal (r-z) section of a two-layer wire-coating die land (not to scale)."""
    C_IN, C_OUT, C_WIRE, C_DIE = "#2a78d6", "#eb6834", "#b0b0b0", "#4a4a48"
    Rw, r1, Rd = 1.0, 1.55, 2.3                 # drawing units, not to scale
    z_in, z_out, z_end = 2.0, 8.0, 11.5         # die land runs from z_in to z_out
    Rf1, Rf = 1.32, 1.80                        # final coating radii on the wire (after the die)
    fig, ax = plt.subplots(figsize=(12, 6.2))
    for sgn in (1, -1):                         # draw the upper half, then mirror it
        # die body (dark), with a converging entry on the left
        ax.add_patch(Polygon([(0.2, sgn*4.0), (z_in, sgn*Rd), (z_out, sgn*Rd), (z_out, sgn*3.6),
                              (0.2, sgn*4.6)], closed=True, color=C_DIE))
        # melt layers inside the land (fully developed region modelled here)
        ax.add_patch(Rectangle((z_in, sgn*r1 if sgn > 0 else -Rd), z_out - z_in, Rd - r1, color=C_OUT, alpha=0.85))
        ax.add_patch(Rectangle((z_in, sgn*Rw if sgn > 0 else -r1), z_out - z_in, r1 - Rw, color=C_IN, alpha=0.85))
        # melt streams arriving from the feed channels (left)
        ax.add_patch(Polygon([(0.2, sgn*1.0), (z_in, sgn*Rw), (z_in, sgn*r1), (0.2, sgn*2.2)], color=C_IN, alpha=0.45))
        ax.add_patch(Polygon([(0.2, sgn*2.2), (z_in, sgn*r1), (z_in, sgn*Rd), (0.2, sgn*4.0)], color=C_OUT, alpha=0.45))
        # after the exit the coating accelerates to wire speed and thins to its final thickness
        zz = np.linspace(z_out, z_end, 60); blend = 1 - np.exp(-(zz - z_out) / 0.45)
        rr1 = r1 + (Rf1 - r1) * blend; rro = Rd + (Rf - Rd) * blend
        ax.fill_between(zz, sgn*Rw, sgn*rr1, color=C_IN, alpha=0.85, lw=0)
        ax.fill_between(zz, sgn*rr1, sgn*rro, color=C_OUT, alpha=0.85, lw=0)
    ax.add_patch(Rectangle((-0.3, -Rw), z_end + 0.6, 2*Rw, color=C_WIRE))          # the wire
    ax.plot([-0.3, z_end + 0.3], [0, 0], color="0.3", lw=0.8, ls="-.")             # axis of symmetry
    # wire motion
    ax.add_patch(FancyArrowPatch((8.6, 0.0), (10.8, 0.0), arrowstyle="-|>", mutation_scale=18, lw=2, color="0.1"))
    ax.text(9.7, 0.25, r"wire, speed $V_w$", ha="center", fontsize=10)
    # velocity profile sketch in the land (upper half), arrows from the wire to the die
    z0 = 5.6
    for r in np.linspace(Rw + 0.06, Rd - 0.06, 7):
        L = 1.9 * (1 - (r - Rw) / (Rd - Rw)) ** 0.7 + 0.05
        ax.add_patch(FancyArrowPatch((z0, r), (z0 + L, r), arrowstyle="-|>", mutation_scale=10, lw=1.1, color="0.05"))
    ax.plot([z0, z0], [Rw, Rd], color="0.05", lw=1.1)
    ax.text(z0 + 0.9, 2.75, r"velocity profile $u(r)$", fontsize=10, ha="center", color="white")
    # radii labels (lower half, at the right end of the land)
    for rv, lab in ((Rw, r"$R_w$"), (r1, r"$r_1$"), (Rd, r"$R_d$")):
        ax.annotate("", xy=(7.4, -rv), xytext=(7.4, 0), arrowprops=dict(arrowstyle="->", lw=1, color="0.1"))
        ax.text(7.48, -rv + 0.12, lab, fontsize=11, va="center")
    # land length
    ax.annotate("", xy=(z_in, -4.95), xytext=(z_out, -4.95), arrowprops=dict(arrowstyle="<->", lw=1, color="0.1"))
    ax.text(5.0, -5.35, r"die land, length $L$ (fully developed flow — the region modelled)", ha="center", fontsize=10)
    ax.text(1.0, 4.9, "melt from the\nextruders", ha="center", fontsize=9)
    ax.text(9.9, 2.9, "final coating: cools and\nmoves with the wire", ha="center", fontsize=9.5)
    ax.text(2.2, 1.27, "inner layer (e.g. insulation)", ha="left", va="center", fontsize=9.5, color="white")
    ax.text(2.2, 1.93, "outer layer (e.g. jacket)", ha="left", va="center", fontsize=9.5, color="white")
    ax.text(3.0, 3.25, "die", ha="center", fontsize=11, color="white")
    # coordinate arrows
    ax.annotate("", xy=(1.3, -4.1), xytext=(0.3, -4.1), arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.text(1.35, -4.1, r"$z$", va="center", fontsize=11)
    ax.annotate("", xy=(0.3, -3.1), xytext=(0.3, -4.1), arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.text(0.3, -2.95, r"$r$", ha="center", fontsize=11)
    ax.set_xlim(-0.4, z_end + 0.4); ax.set_ylim(-5.7, 5.4)
    ax.axis("off")
    ax.set_title("Two-layer wire coating: longitudinal section through the die (not to scale)", fontsize=12)
    fig.tight_layout(); plt.show()

draw_wire_coating_schematic()

Inside the land, the inner layer (blue) is in contact with the moving wire, and the outer layer (orange) with the stationary die. After the exit, each layer accelerates or decelerates to the wire speed and thins to its **final coating thickness**. Because the final coating moves at the wire speed, the final thicknesses fix the flow rates (Section 2.7).

Typical engineering questions this model answers:
* What **die pressure** is needed to put a given coating on the wire at a given line speed?
* How large is the **pulling force** (wire tension) contributed by the land?
* Where does the **interface** sit inside the die, and how large is the **shear-rate jump** across it? Large jumps are associated with interfacial instabilities in coextrusion.
* Are the shear stresses approaching the levels where surface defects such as *sharkskin* appear?

### 1.3 Nomenclature

| Symbol | Meaning | SI units |
|---|---|---|
| $r, z$ | radial and axial coordinates (flow and wire motion in $+z$) | m |
| $R_w$, $R_d$ | wire radius, die-land radius | m |
| $h = R_d - R_w$ | gap | m |
| $r_1$ ($r_i$) | interface radius (between layers $i$ and $i+1$) | m |
| $L$ | die-land length | m |
| $V_w$, $V_d$ | wire speed (line speed), die-wall speed (here 0) | m/s |
| $u(r)$ | axial velocity $u_z$ | m/s |
| $\dot\gamma = du/dr$ | shear rate | 1/s |
| $\tau = \tau_{rz}$ | shear stress | Pa |
| $\tau_w, \tau_d$ | shear stress at the wire and at the die wall | Pa |
| $p$ | pressure | Pa |
| $G = -dp/dz$ | pressure gradient (positive when pressure falls in the flow direction) | Pa/m |
| $C$ | integration constant in the stress distribution | Pa·m |
| $r^*$ | zero-stress radius, $\tau(r^*) = 0$ | m |
| $K_i$, $n_i$ | power-law consistency and index of layer $i$ | Pa·sⁿ, — |
| $s_i = 1/n_i$ | reciprocal power-law index | — |
| $\eta$ | (apparent) viscosity $= K|\dot\gamma|^{n-1}$ | Pa·s |
| $Q_i$ | volumetric flow rate of layer $i$ | m³/s |
| $t_i$, $R_i$ | final coating thickness and final outer radius of layer $i$ (after the die) | m |
| $F$ | axial force of the melt on the wire over the land | N |
| $\rho$, $c_p$ | melt density, heat capacity | kg/m³, J/(kg K) |

<a id="part2"></a>
# 2. Model formulation

### 2.1 Assumptions

| # | Assumption | Why it is reasonable (or when it fails) |
|---|---|---|
| A1 | **Steady**, **laminar**, **creeping** flow (inertia negligible) | Melts are very viscous, so the Reynolds number is of order $10^{-2}$ (estimated below). |
| A2 | **Incompressible**, **isothermal** melts | The pressures here are a few MPa, so density changes are below 1 %. Viscous heating is estimated after Example 1 (about 1 K there); it grows with line speed and land length. |
| A3 | **Fully developed** flow: $u = u_z(r)$ only, and $u_r = u_\theta = 0$ | The entrance length scales with Re·$h$, which is tiny, so the land is fully developed almost everywhere. |
| A4 | **Concentric** wire, **axisymmetric** flow | This needs a well-centred wire. Eccentricity gives non-uniform coatings and is not modelled. |
| A5 | **No slip** at the wire and at the die wall | This is the standard assumption. Wall slip can occur at high stress; see the companion notebook with slip. |
| A6 | Flat, cylindrical, **stable interfaces**, with continuous velocity and shear stress across them | This holds for well-bonded layers below the onset of interfacial instability. |
| A7 | **Power-law** viscosity $\eta = K|\dot\gamma|^{n-1}$ in each layer | The power law fits melts well at high shear rates. It fails at low shear rates, near $r^*$ or in nearly stagnant regions, where real melts have a Newtonian plateau (the Carreau notebook covers this). |
| A8 | **Gravity** negligible | $\rho g$ is about $10^4$ Pa/m, compared with pressure gradients of $10^8$ Pa/m. |

The cell below puts numbers on A1, A3 and A8 for typical conditions.

In [ ]:
# Order-of-magnitude check of the assumptions, for typical wire-coating conditions
rho   = 900.0      # melt density [kg/m^3] (polyolefins are about 750-950 kg/m^3 in the melt)
V     = 1.0        # line speed [m/s]
h     = 0.5e-3     # gap R_d - R_w [m]
K, n  = 4000.0, 0.40   # power-law parameters [Pa s^n], [-]

gdot  = V / h                          # characteristic shear rate [1/s]
eta   = K * gdot**(n - 1)              # apparent viscosity at that rate [Pa s]
Re    = rho * V * h / eta              # Reynolds number
L_ent = 0.06 * Re * h                  # rough laminar entrance length (Newtonian estimate) [m]
grad_typ = 1e8                         # typical pressure gradient in a land [Pa/m] (Example 1 gives ~3e8)

print(f"characteristic shear rate   V/h      = {gdot:8.0f} 1/s")
print(f"apparent viscosity          eta      = {eta:8.1f} Pa s")
print(f"Reynolds number             Re       = {Re:8.4f}    -> creeping flow (Re << 1)")
print(f"entrance length             ~0.06 Re h = {L_ent*1e6:6.3f} um  -> fully developed almost at once")
print(f"gravity / pressure gradient rho g / G = {rho*9.81/grad_typ:.0e}   -> gravity negligible")

### 2.2 Kinematics and continuity

With $u_r = u_\theta = 0$ and $u_z = u(r)$ (A3, A4), the continuity equation for an incompressible fluid,
$$\frac{1}{r}\frac{\partial (r u_r)}{\partial r} + \frac{1}{r}\frac{\partial u_\theta}{\partial \theta} + \frac{\partial u_z}{\partial z} = 0,$$
is satisfied identically. The only non-zero velocity gradient is $du/dr$, so the only non-zero shear stress is $\tau_{rz} = \tau_{zr}$, which we write as $\tau(r)$.

### 2.3 Momentum balance: the stress distribution

**Radial momentum.** With $u_r = 0$, the $r$-component of the momentum equation reduces to
$$\frac{\partial p}{\partial r} = 0 . \tag{2.1}$$
The pressure therefore depends on $z$ only, and **$dp/dz$ is the same at every radius, in every layer**.

**Shell balance in $z$.** Take a thin cylindrical shell of radius $r$, thickness $\Delta r$ and length $\Delta z$. In steady, fully developed flow, momentum does not change along $z$, so the axial forces must balance:
$$\underbrace{\big[p(z) - p(z+\Delta z)\big]\,2\pi r\,\Delta r}_{\text{pressure}} \;+\; \underbrace{\big[(2\pi r\,\tau)\big|_{r+\Delta r} - (2\pi r\,\tau)\big|_{r}\big]\Delta z}_{\text{shear on the inner and outer surfaces}} = 0 .$$
Dividing by $2\pi\,\Delta r\,\Delta z$ and letting both go to zero gives
$$\frac{1}{r}\frac{d}{dr}\big(r\,\tau\big) = \frac{dp}{dz} \equiv -G. \tag{2.2}$$
Here $\tau$ is $\tau_{rz}$ in the convention $\tau = +\eta\,du/dr$ (shear stress *exerted by the fluid at larger r* on the fluid at smaller $r$). With this convention Eq. 2.2 has the plus sign shown. BSL use the opposite (momentum-flux) sign convention; the physics is the same.

**Integrating** Eq. 2.2 once:
$$r\,\tau = -\frac{G r^2}{2} + C \quad\Longrightarrow\quad \boxed{\;\tau(r) = -\frac{G\,r}{2} + \frac{C}{r}\;} \tag{2.3}$$

**Why one pair $(G, C)$ holds in all layers.** $G$ is the same everywhere because $p = p(z)$. At an interface, a force balance on a vanishingly thin shell requires the shear stress, and hence $r\tau$, to be continuous. Integrating Eq. 2.2 across an interface therefore does not change $C$. The whole multilayer problem has **only two stress constants**, plus the interface positions.

Compare a **slit** (flat channel): there $d\tau/dy = dp/dx$ and the stress is a straight line. The $C/r$ term is the **curvature effect** of the annulus. It matters when the gap is not small compared with the wire radius, which is common in wire coating.

<details><summary><b>▸ Check your understanding:</b> Where is $|\tau|$ largest for pure drag flow ($G = 0$)?</summary>

With $G = 0$, $\tau = C/r$, so $|\tau|$ is largest at the **wire** (smallest $r$). It is larger there than at the die wall by the factor $R_d/R_w$. Physically, the same axial force $2\pi r L\,\tau$ is transmitted through every cylindrical surface, and the inner surfaces are smaller.
</details>

### 2.4 The power-law fluid

For a generalized Newtonian fluid in this simple shear flow, $\tau = \eta(\dot\gamma)\,\dot\gamma$ with $\dot\gamma = du/dr$. The **power law** (Ostwald–de Waele) is
$$\tau = K\,|\dot\gamma|^{\,n-1}\,\dot\gamma, \qquad \eta = K|\dot\gamma|^{\,n-1}. \tag{2.4}$$
* $n < 1$: **shear thinning**, as in polymer melts (typically $n \approx 0.2$–$0.6$). $n = 1$ gives a Newtonian fluid with $\mu = K$.
* $K$ is the stress at $\dot\gamma = 1\ \text{s}^{-1}$. Its units, Pa·sⁿ, depend on $n$.
* The form $|\dot\gamma|^{n-1}\dot\gamma$ (not $\dot\gamma^n$) keeps the stress and the shear rate with the **same sign**. This matters here, because $du/dr$ can be positive in one part of the gap and negative in another.

Equation 2.4 is easily inverted to give the shear rate for a known stress:
$$\frac{du}{dr} = \dot\gamma(\tau) = \operatorname{sign}(\tau)\left(\frac{|\tau|}{K}\right)^{1/n}. \tag{2.5}$$
Since $\tau(r)$ is known from Eq. 2.3 once $G$ and $C$ are known, Eq. 2.5 gives the shear rate at every radius.

<details><summary><b>▸ Check your understanding:</b> Why is the power-law viscosity infinite at the zero-stress radius, and is that physical?</summary>

At $r^*$, $\dot\gamma = 0$, and $\eta = K|\dot\gamma|^{n-1} \to \infty$ for $n < 1$. Real melts have a finite zero-shear viscosity $\eta_0$, so this is an artefact of the power law. It has little effect on the flow rates, because the shear rate (not the viscosity) is what gets integrated, and that stays finite. It is still the main reason to use a Carreau model when low-shear regions matter.
</details>

### 2.5 Boundary and interface conditions

$$u(R_w) = V_w \quad\text{(no slip at the wire)}, \qquad u(R_d) = V_d = 0 \quad\text{(no slip at the die)} \tag{2.6a}$$
$$u \text{ continuous at each interface } r_i, \qquad \tau \text{ continuous at each interface } r_i. \tag{2.6b}$$
Stress continuity is already built into Eq. 2.3.

### 2.6 Velocity and flow rate in each layer

**Velocity.** Integrating Eq. 2.5 outward from the wire, one layer at a time:
$$u(r) = u(r_{i-1}) + \int_{r_{i-1}}^{r}\dot\gamma_i\big(\tau(r')\big)\,dr', \qquad r_{i-1}\le r \le r_i, \qquad u(r_0) = u(R_w) = V_w . \tag{2.7}$$
Each layer starts from the velocity at the end of the previous one, so velocity continuity (Eq. 2.6b) is automatic.

**Flow rate.** The flow rate of layer $i$ (between $a = r_{i-1}$ and $b = r_i$) is $Q_i = 2\pi\int_a^b u\,r\,dr$. Write $u(r) = u(a) + [u(r) - u(a)]$ and integrate the second part **by parts**, using $d[u - u(a)]/dr = \dot\gamma$ and $\int r\,dr = -\tfrac12(b^2 - r^2)$:
$$\int_a^b [u(r) - u(a)]\,r\,dr = \Big[-\tfrac12(b^2 - r^2)\,(u - u(a))\Big]_a^b + \int_a^b \dot\gamma(r)\,\tfrac12(b^2 - r^2)\,dr .$$
The bracket vanishes at both limits, which leaves
$$Q_i = 2\pi\left[\tfrac12\big(b^2 - a^2\big)\,u(a) + \int_a^b \dot\gamma_i(r)\,\tfrac12\big(b^2 - r^2\big)\,dr\right]. \tag{2.8}$$
This is a **single** integral of the shear rate, so we never need $u(r)$ at every point to get $Q_i$.

### 2.7 The wire-coating mass balance and the wire force

After the die the coating moves at the wire speed (plug flow). If $R_k$ is the final outer radius of layer $k$ (with $R_0 = R_w$), mass conservation requires
$$Q_k = V_w\,\pi\left(R_k^2 - R_{k-1}^2\right). \tag{2.9}$$
So **specifying final coating thicknesses is the same as specifying flow rates**. Die swell and draw-down are neglected.

The axial **force of the melt on the wire** over a land of length $L$ is the wire-surface shear stress times the wetted area:
$$F = 2\pi R_w L\,\tau_w. \tag{2.10}$$
It is negative when the melt resists the wire, which is the usual case. The puller must supply $-F$, in addition to other line tensions.

### 2.8 Counting unknowns and equations

| Unknowns ($N+1$) | Equations ($N+1$) |
|---|---|
| $G$ and $C$ (equivalently the wall stresses $\tau_w$, $\tau_d$) | $N$ flow rates: $Q_i(\text{calculated}) = Q_i(\text{given})$ |
| $N - 1$ interface radii $r_1, \dots, r_{N-1}$ | 1 boundary condition: $u(R_d) = V_d$ (from marching Eq. 2.7 out from the wire) |

For two layers that is **3 unknowns and 3 equations**. The equations are **nonlinear**: $\dot\gamma \propto |\tau|^{1/n}$, and the interface positions appear in the integration limits. They are solved numerically (Section 3).

In the code the unknowns are the **two wall stresses** rather than $(G, C)$. They are better scaled (both are of order $10^4$–$10^5$ Pa), and they are linked to $(G, C)$ by writing Eq. 2.3 at $r = R_w$ and $r = R_d$:
$$G = \frac{2\,(R_w\tau_w - R_d\tau_d)}{R_d^2 - R_w^2}, \qquad C = R_w\tau_w + \tfrac12 G R_w^2 . \tag{2.11}$$

### 2.9 The zero-stress radius and the three flow regimes

Setting $\tau(r^*) = 0$ in Eq. 2.3 gives $r^{*2} = 2C/G$. A real $r^*$ exists only if $C/G > 0$. When $r^*$ lies **inside** the annulus, the velocity has a maximum or minimum there and the shear rate changes sign. The sign of $G$ classifies the flow:

| Regime | $G = -dp/dz$ | What is happening |
|---|---|---|
| Pressure-assisted | $G > 0$ | The die pushes melt out faster than the wire alone would drag it (thick coatings). |
| Pure drag | $G = 0$ | The wire alone carries exactly the required flow; $\tau = C/r$. |
| Pressure build-up | $G < 0$ | The wire drags more than is delivered, so pressure rises toward the exit and holds melt back (thin coatings). Backflow is possible near the die wall. |

### 2.10 Exact solutions used for checking

* **Newtonian fluid** ($n = 1$, $K = \mu$), one layer, no slip:
$$u(r) = \frac{G}{4\mu}\left[(R_d^2 - r^2) - (R_d^2 - R_w^2)\frac{\ln(R_d/r)}{\ln(R_d/R_w)}\right] + V_w\frac{\ln(R_d/r)}{\ln(R_d/R_w)} . \tag{2.12}$$
* **Newtonian pure drag** ($G = 0$; BSL Problem 2B.7, with $\kappa = R_w/R_d$):
$$Q = \frac{\pi R_d^2 V_w}{2}\left[\frac{1-\kappa^2}{\ln(1/\kappa)} - 2\kappa^2\right], \qquad \tau_w = -\frac{\mu V_w}{R_w\ln(R_d/R_w)} . \tag{2.13}$$
* **Power-law pure drag** ($G = 0$, $\tau = C/r$ with $C < 0$), from Eqs. 2.5 and 2.7:
$$u(r) = V_w - \left(\frac{|C|}{K}\right)^{s}\frac{r^{1-s} - R_w^{1-s}}{1-s}, \qquad s = 1/n \ne 1 . \tag{2.14}$$

<a id="part3"></a>
# 3. Numerical method

### 3.1 Why a numerical solution?

For a **slit**, the power-law velocity in each layer can be integrated in closed form. In an **annulus** the integrand in Eq. 2.7 is
$$\dot\gamma(r) \propto \left|-\frac{G r}{2} + \frac{C}{r}\right|^{1/n}\operatorname{sign}(\cdot),$$
and its integral has no elementary closed form for general $n$; it leads to hypergeometric functions. Instead we evaluate the integrals in Eqs. 2.7 and 2.8 by **numerical quadrature**, accurately enough that the result is exact to round-off for practical purposes, and solve the $N+1$ nonlinear equations iteratively.

### 3.2 Quadrature: Gauss–Legendre, and why the zero-stress radius needs care

**Gauss–Legendre quadrature** approximates $\int_a^b f(r)\,dr \approx \sum_k w_k f(r_k)$ with carefully chosen nodes $r_k$ and weights $w_k$. With $m$ nodes it is *exact* for polynomials of degree up to $2m - 1$, and it converges extremely fast for **smooth** functions. We use $m = 48$.

**The difficulty.** Near the zero-stress radius $r^*$ the shear rate behaves like $|r - r^*|^{1/n}$. For $n = 0.4$ that is $|r - r^*|^{2.5}$: continuous, but its higher derivatives are infinite at $r^*$. Polynomial-based quadrature then converges slowly. The trouble appears both when $r^*$ is inside a layer and when it lies just outside one.

**The fix.** The stress can be factorized exactly around its zero:
$$\tau(r) = -\frac{G}{2}\,\frac{(r - r^*)(r + r^*)}{r} \quad\Longrightarrow\quad \dot\gamma(r) = \underbrace{|r - r^*|^{1/n}}_{\text{non-smooth}}\;\times\;\underbrace{\left(-\operatorname{sign}G\,\operatorname{sign}(r - r^*)\left(\frac{|G|}{2K}\right)^{1/n}\left(\frac{r + r^*}{r}\right)^{1/n}\right)}_{\text{smooth}} \tag{3.1}$$
**Gauss–Jacobi quadrature** is designed for integrals of the form $\int (1+x)^{\beta} f(x)\,dx$. It treats the factor $(1+x)^\beta$ as a *weight* and integrates it exactly, so only the smooth part $f$ is sampled. With $\beta = 1/n$, every integral is computed from $r^*$ outward, and an integral over a layer is a difference of two such integrals. This is used whenever $r^*$ lies inside a layer or within one layer-thickness of it. Otherwise ordinary Gauss–Legendre is used. Verification test 5 confirms round-off accuracy.

### 3.3 Solving the nonlinear equations

Collect the unknowns in a vector $\mathbf{x}$ and the equation errors in a **residual vector**
$$\mathbf{R}(\mathbf{x}) = \left[\frac{Q_1^{\text{calc}} - Q_1}{Q_1},\ \dots,\ \frac{Q_N^{\text{calc}} - Q_N}{Q_N},\ \frac{u(R_d) - V_d}{U}\right],$$
where every entry is made **dimensionless** so that all equations count equally ($U$ is a velocity scale). The solution satisfies $\mathbf{R} = \mathbf{0}$. We use `scipy.optimize.least_squares`, a trust-region method that minimizes $\tfrac12\|\mathbf{R}\|^2$. At the true solution that minimum is zero, typically about $10^{-15}$ in practice, which is the *solver residual* printed in the reports.

Two **changes of variable** make the problem easier for the solver:
* **Wall stresses through $\operatorname{asinh}$:** $\tau = \tau_\text{ref}\sinh(x)$. This behaves like a logarithm for large $|\tau|$, covering many decades evenly, but passes smoothly through zero, so either sign is allowed.
* **Layer thicknesses through "softmax" weights:** thickness $\propto e^{w_i}$, normalized to add up to the gap. Any $\mathbf{w}$ gives positive thicknesses that fill the gap exactly, so the interfaces can never cross or leave the annulus.

### 3.4 Solution strategy: start well, then continue if needed

Newton-type solvers need a reasonable starting point. The solver therefore works in up to three stages:

1. **Exact uniform-fluid start.** It replaces all layers by one "average" fluid and solves that problem *exactly*, using two nested one-dimensional root finds that are guaranteed to converge. Two monotonic properties make this possible: for fixed $G$, the die-wall velocity increases with $C$; and the flow rate increases with $G$. It then places the interfaces where the cumulative flow reaches each layer's share. From this start, the real problem usually converges in a few iterations.
2. **Continuation.** If the layers differ strongly (for example, a viscosity ratio of 100), it moves $K_i$ and $n_i$ gradually from the average values to the real ones, re-solving at each step from the previous answer. If a step fails, the step is halved.
3. **Multi-start.** As a last resort, it tries many random starting points, screens them cheaply, and polishes the best.

### 3.5 How we know the answer is right

Before using a numerical model for design, it must be **verified**: shown to solve its equations correctly. In Part 5 we compare it with the exact solutions of Section 2.10, check internal consistency (one fluid split into two "layers" must give the same answer as one layer), and re-integrate the converged solution with an independent adaptive quadrature routine. Verification is different from **validation**, which compares with experiments and tests the assumptions themselves.

<a id="part4"></a>
# 4. The code, cell by cell

The solver is split into short cells, each preceded by a description. **Run each cell in order.** Cells 4.1–4.9 only define functions and print nothing; cell 4.10 makes a first call to the solver, and the systematic calculations start in Part 5.

### 4.1 Libraries

`numpy` handles arrays and vectorized math. From `scipy` we use `brentq`, a robust 1-D root finder; `least_squares`, the nonlinear solver; and `roots_jacobi`, which supplies Gauss–Jacobi nodes and weights.

In [ ]:
# ---------------------------------------------------------------------------
# Libraries
#   numpy      : arrays and vectorized math
#   scipy      : root finding (brentq), nonlinear least squares, Gauss-Jacobi nodes
#   matplotlib : plots (imported later, in the plotting cell)
# ---------------------------------------------------------------------------
import numpy as np
from functools import lru_cache
from scipy.optimize import least_squares, brentq
from scipy.special import roots_jacobi

MAX_LAYERS = 5      # the solver accepts 1-5 concentric layers; the examples use 2

### 4.2 Stress, shear rate and the zero-stress radius

These three short functions are Eqs. 2.3, 2.5 and the definition of $r^*$ (Section 2.9). Everything else is built on them. They accept numpy arrays, so a whole set of radii can be evaluated in one call.

In [ ]:
# ---------------------------------------------------------------------------
# Shear stress and shear rate (Sections 2.3 and 2.4 of the tutorial)
# ---------------------------------------------------------------------------
def shear_stress(r, G, C):
    """Shear stress tau_rz [Pa] at radius r [m].

    From the integrated z-momentum balance (Eq. 2.3):
        tau_rz(r) = -G r / 2 + C / r
    G = -dp/dz [Pa/m] and C [Pa m] are the two integration constants.
    They are the same in every layer, because r*tau_rz is continuous at
    each interface. Works for scalar r or numpy arrays."""
    return -0.5 * G * r + C / r


def shear_rate(tau, K, n):
    """Shear rate du/dr [1/s] produced by shear stress tau [Pa] in a power-law fluid.

    Power law (Eq. 2.4):  tau = K |du/dr|^(n-1) du/dr
    Inverted (Eq. 2.5):   du/dr = sign(tau) * (|tau| / K)^(1/n)
    The sign is kept so that du/dr has the same sign as tau."""
    return np.sign(tau) * np.abs(tau / K) ** (1.0 / n)


def zero_stress_radius(G, C):
    """Radius r* [m] where tau_rz = 0, or None if there is no such radius.

    Setting -G r/2 + C/r = 0 gives r*^2 = 2C/G, which is real only if C/G > 0.
    r* may lie inside or outside the annulus; the solver handles both cases.
    When r* is inside, it is the radius of maximum (or minimum) velocity."""
    if G != 0 and C / G > 0:
        return np.sqrt(2.0 * C / G)
    return None

### 4.3 Integrating across one layer

`layer_integrals(a, b, G, C, K, n)` returns the two integrals needed for a layer between radii $a$ and $b$:
* `du` $= u(b) - u(a) = \int_a^b \dot\gamma\,dr$, from Eq. 2.7;
* `dq` $= \int_a^b \dot\gamma\,\tfrac12(b^2 - r^2)\,dr$, the integral in Eq. 2.8.

It chooses the quadrature rule as described in Section 3.2: Gauss–Jacobi anchored at $r^*$ when $r^*$ is inside or near the layer (using the factorization in Eq. 3.1), and Gauss–Legendre otherwise.

In [ ]:
# ---------------------------------------------------------------------------
# Numerical integration across one layer (Section 3.2 of the tutorial)
#
# Each layer needs two integrals of the shear rate g(r) = du/dr over [a, b]:
#   (1)  u(b) - u(a)                 = int_a^b g(r) dr
#   (2)  int_a^b (u - u(a)) r dr     = int_a^b g(r) * (b^2 - r^2)/2 dr   (integration by parts)
#
# Two quadrature rules are used:
#   * Gauss-Legendre (48 points) when g(r) is smooth over the layer.
#   * Gauss-Jacobi near the zero-stress radius r*, where g ~ |r - r*|^(1/n)
#     is not smooth (its derivatives blow up for non-integer 1/n). Writing
#         g(r) = |r - r*|^s * (smooth function),   s = 1/n
#     and using |r - r*|^s as the quadrature WEIGHT integrates the
#     non-smooth part exactly, so the result is accurate to round-off.
# ---------------------------------------------------------------------------
N_LEGENDRE = 48
N_JACOBI = 40
_X_LEG, _W_LEG = np.polynomial.legendre.leggauss(N_LEGENDRE)   # nodes/weights on [-1, 1]


@lru_cache(maxsize=4096)
def _jacobi_rule(s):
    """Gauss-Jacobi nodes/weights on [-1, 1] for the weight (1 + x)^s (cached per s)."""
    return roots_jacobi(N_JACOBI, 0.0, s)


def _integral_from_r_star(x, r_star, G, K, s, weight):
    """Integral of g(r)*weight(r) from r* to x, with g = du/dr, by Gauss-Jacobi.

    Uses the exact factorization of the stress around its zero (Eq. 3.1):
        tau(r) = -(G/2) (r - r*) (r + r*) / r
    so that
        g(r) = -sign(G) sign(r - r*) (|G|/(2K))^s |r - r*|^s ((r + r*)/r)^s.
    Everything except |r - r*|^s is smooth and is evaluated at the nodes."""
    if x == r_star:
        return 0.0
    t, w = _jacobi_rule(s)
    d = x - r_star                                   # signed length of the interval
    r = r_star + 0.5 * d * (1.0 + t)                 # map nodes from [-1, 1] to [r*, x]
    smooth_part = (-np.sign(G) * np.sign(d) * (abs(G) / (2.0 * K)) ** s
                   * ((r + r_star) / r) ** s * weight(r))
    # |r - r*|^s = (|d|/2)^s (1 + t)^s   and   dr = (d/2) dt
    return (abs(d) / 2.0) ** s * (d / 2.0) * np.sum(w * smooth_part)


def layer_integrals(a, b, G, C, K, n):
    """Integrals over one power-law layer occupying a <= r <= b.

    Returns (du, dq) with
        du = u(b) - u(a)                        [m/s]
        dq = int_a^b (u(r) - u(a)) r dr         [m^3/s per radian]
    The layer flow rate then follows from Eq. 2.8 (see march_through_layers)."""
    s = 1.0 / n
    weight_u = lambda r: np.ones_like(r)                 # for du
    weight_q = lambda r: 0.5 * (b * b - r * r)           # for dq (integration by parts)
    r_star = zero_stress_radius(G, C)
    L = b - a
    if r_star is not None and (a - L) < r_star < (b + L):
        # r* is inside the layer or close to it: integrate from r* with Gauss-Jacobi
        # and take differences, int_a^b = int_{r*}^b - int_{r*}^a.
        du = (_integral_from_r_star(b, r_star, G, K, s, weight_u)
              - _integral_from_r_star(a, r_star, G, K, s, weight_u))
        dq = (_integral_from_r_star(b, r_star, G, K, s, weight_q)
              - _integral_from_r_star(a, r_star, G, K, s, weight_q))
        return du, dq
    # r* is absent or far away: the integrand is smooth, so use Gauss-Legendre.
    r = 0.5 * L * _X_LEG + 0.5 * (a + b)                 # nodes mapped to [a, b]
    w = 0.5 * L * _W_LEG
    g = shear_rate(shear_stress(r, G, C), K, n)
    return np.sum(w * g), np.sum(w * g * weight_q(r))

### 4.4 Marching through the layers

`march_through_layers` applies Eqs. 2.7 and 2.8 layer by layer, starting from $u(R_w) = V_w$. Given trial values of $G$, $C$ and the interface radii, it returns the velocity at every interface and the flow rate of every layer. The solver compares these with the required flow rates and with $u(R_d) = 0$.

`G_and_C_from_wall_stresses` is Eq. 2.11.

In [ ]:
# ---------------------------------------------------------------------------
# Marching outward through the layers (Sections 2.6 and 2.8 of the tutorial)
# ---------------------------------------------------------------------------
def march_through_layers(G, C, radii, K, n, V_wire):
    """Velocity at every interface and flow rate of every layer, for given G, C and radii.

    Starts from no slip at the wire, u(R_w) = V_wire, and integrates outward
    one layer at a time. Velocity continuity at each interface is automatic,
    because each layer starts from the velocity at the end of the previous one.

    radii : array [R_w, r_1, ..., R_d] of length N + 1
    Returns (u_edges, Q_layers):
        u_edges[i]  = velocity at radii[i]                  [m/s]
        Q_layers[i] = volumetric flow rate of layer i       [m^3/s]"""
    N = len(K)
    u_edges = np.empty(N + 1)
    u_edges[0] = V_wire
    Q_layers = np.zeros(N)
    for i in range(N):
        a, b = radii[i], radii[i + 1]
        du, dq = layer_integrals(a, b, G, C, K[i], n[i])
        # Eq. 2.8:  Q_i = 2 pi [ u(a) (b^2 - a^2)/2 + int_a^b (u - u(a)) r dr ]
        Q_layers[i] = 2.0 * np.pi * (0.5 * (b * b - a * a) * u_edges[i] + dq)
        u_edges[i + 1] = u_edges[i] + du
    return u_edges, Q_layers


def G_and_C_from_wall_stresses(tau_wire, tau_die, R_wire, R_die):
    """Convert the two wall shear stresses into the constants G and C (Eq. 2.11).

    Writing Eq. 2.3 at r = R_w and at r = R_d gives two linear equations:
        R_w tau_w = -G R_w^2 / 2 + C
        R_d tau_d = -G R_d^2 / 2 + C"""
    G = 2.0 * (R_wire * tau_wire - R_die * tau_die) / (R_die ** 2 - R_wire ** 2)
    C = R_wire * tau_wire + 0.5 * G * R_wire ** 2
    return G, C

### 4.5 Flow rates from final coating thicknesses

`flow_rates_from_coating` is Eq. 2.9. This is usually how a wire-coating problem is posed: *"put 0.10 mm of insulation and 0.20 mm of jacket on a 1 mm wire at 1 m/s"*.

In [ ]:
# ---------------------------------------------------------------------------
# Wire-coating mass balance (Section 2.7 of the tutorial)
# ---------------------------------------------------------------------------
def flow_rates_from_coating(R_wire, coating_thickness, V_wire):
    """Flow rate of each layer [m^3/s] needed for the requested FINAL coating.

    Downstream of the die the solidifying coating moves with the wire, so layer k
    carries Q_k = V_wire * pi * (R_k^2 - R_{k-1}^2), where R_k is the final outer
    radius of layer k on the wire (die swell and draw-down are neglected).

    coating_thickness : list of final layer thicknesses [m], from the wire outward."""
    R = R_wire + np.concatenate([[0.0], np.cumsum(coating_thickness)])
    return V_wire * np.pi * np.diff(R ** 2)

### 4.6 The uniform-fluid starting solution

These helpers implement stage 1 of Section 3.4. `_bracket_root` widens an interval until an increasing function changes sign, then calls `brentq`. `_uniform_fluid_solution` uses two nested root finds to get $G$ and $C$ for a single fluid, then locates the interfaces from the cumulative flow. You do not need to call these yourself; `solve_annular` uses them.

In [ ]:
# ---------------------------------------------------------------------------
# Helper: an exact solution for ONE uniform fluid, used as the starting guess
# for the real multilayer problem (Section 3.4 of the tutorial)
# ---------------------------------------------------------------------------
def _bracket_root(f, x0, step=1.0, grow=1.6, maxit=200):
    """Find a root of an INCREASING function f: widen [a, b] around x0 until f
    changes sign, then use Brent's method (scipy.optimize.brentq)."""
    a, b = x0 - step, x0 + step
    fa, fb = f(a), f(b)
    for _ in range(maxit):
        if fa <= 0 <= fb:
            break
        if fa > 0:                 # root lies to the left: move the bracket left
            b, fb = a, fa
            step *= grow; a -= step; fa = f(a)
        else:                      # root lies to the right: move the bracket right
            a, fa = b, fb
            step *= grow; b += step; fb = f(b)
    else:
        raise RuntimeError("could not bracket root")
    if fa == 0: return a
    if fb == 0: return b
    return brentq(f, a, b, xtol=1e-13, rtol=1e-15, maxiter=500)


def _uniform_fluid_solution(Q_frac, Q_total, K, n, R_wire, R_die, V_wire, V_die, tau_ref):
    """Solve exactly for a single fluid (K, n) filling the annulus, then place the
    layer interfaces where the cumulative flow from the wire reaches each layer's share.

    Uses two facts:
      * for fixed G, the die-wall velocity u(R_d) increases monotonically with C;
      * the total flow increases monotonically with G = -dp/dz.
    So two nested 1-D root finds, each guaranteed to converge, give G and C.
    Returns (tau_wire, tau_die, radii)."""
    G_ref = tau_ref / (R_die - R_wire)

    def die_velocity_error(C, G):
        du, _ = layer_integrals(R_wire, R_die, G, C, K, n)
        return V_wire + du - V_die

    def C_for_given_G(G):
        z = _bracket_root(lambda x: die_velocity_error(R_wire * tau_ref * np.sinh(x), G) / tau_ref, 0.0)
        return R_wire * tau_ref * np.sinh(z)

    def flow_error(zG):
        G = G_ref * np.sinh(zG)
        C = C_for_given_G(G)
        _, dq = layer_integrals(R_wire, R_die, G, C, K, n)
        Q = 2.0 * np.pi * (0.5 * (R_die ** 2 - R_wire ** 2) * V_wire + dq)
        return (Q - Q_total) / Q_total

    G = G_ref * np.sinh(_bracket_root(flow_error, 0.0))
    C = C_for_given_G(G)

    def cumulative_flow(r):          # flow between the wire and radius r
        if r <= R_wire:
            return 0.0
        _, dq = layer_integrals(R_wire, r, G, C, K, n)
        return 2.0 * np.pi * (0.5 * (r * r - R_wire ** 2) * V_wire + dq)

    r_grid = np.linspace(R_wire, R_die, 2001)
    Q_grid = np.array([cumulative_flow(r) for r in r_grid])
    radii, j0 = [R_wire], 0
    for target in np.cumsum(Q_frac)[:-1] * Q_total:
        j = j0                        # first grid interval (beyond the last interface) that brackets the target
        while j < len(r_grid) - 1 and not ((Q_grid[j] - target) * (Q_grid[j + 1] - target) <= 0
                                           and r_grid[j + 1] > radii[-1]):
            j += 1
        if j == len(r_grid) - 1:
            raise RuntimeError("could not place interface")
        radii.append(brentq(lambda v: cumulative_flow(v) - target,
                            max(r_grid[j], radii[-1]), r_grid[j + 1], xtol=1e-15 * R_die))
        j0 = j
    radii.append(R_die)
    return shear_stress(R_wire, G, C), shear_stress(R_die, G, C), np.array(radii)

### 4.7 The main solver: `solve_annular`

This is the function you call. It:
1. checks the inputs;
2. builds dimensionless scales: a velocity scale $U$, a stress scale $\tau_\text{ref}$, and a flow-weighted "average" fluid;
3. defines the change of variables (`unpack` / `pack`) and the residual vector (Section 3.3);
4. runs the three-stage strategy of Section 3.4;
5. returns the solution as a Python dictionary.

**Keys of the returned dictionary**

| key | meaning | units |
|---|---|---|
| `G`, `dpdz` | $G = -dp/dz$ and $dp/dz$ | Pa/m |
| `C` | constant in Eq. 2.3 | Pa·m |
| `r0` | zero-stress radius $r^*$ (`None` if it does not exist) | m |
| `tau_wire`, `tau_die` | wall shear stresses $\tau_w$, $\tau_d$ | Pa |
| `radii` | $[R_w, r_1, \dots, R_d]$ | m |
| `u_edge` | velocity at each entry of `radii` | m/s |
| `Q`, `Q_calc` | required and calculated flow rates | m³/s |
| `K`, `n`, `R_wire`, `R_die`, `V_wire`, `V_die` | the inputs | — |
| `residual` | size of the remaining equation error (≈ 1e-15 when converged) | — |

In [ ]:
# ---------------------------------------------------------------------------
# The main solver (Sections 2.8 and 3.3-3.4 of the tutorial)
# ---------------------------------------------------------------------------
def solve_annular(Q, K, n, R_wire, R_die, V_wire=0.0, V_die=0.0):
    """Fully developed, isothermal, laminar flow of 1-5 concentric power-law layers
    between a wire (radius R_wire, axial speed V_wire) and a die (radius R_die,
    axial speed V_die, normally 0).

    Inputs (SI units)
        Q       : flow rate of each layer [m^3/s], listed from the wire outward
        K, n    : power-law consistency [Pa s^n] and index [-] of each layer
        R_wire, R_die : radii [m];   V_wire, V_die : wall speeds [m/s]

    Unknowns (N + 1):   tau_wire, tau_die and the N-1 interface radii
    Equations (N + 1):  Q_i(calculated) = Q_i(given) for every layer, and u(R_d) = V_die

    Returns a dict with G (= -dp/dz), C, the zero-stress radius r0, both wall
    stresses, the interface radii, the interface velocities, and more."""
    # ---- check and unpack the inputs ------------------------------------
    Q, K, n = (np.asarray(v, dtype=float) for v in (Q, K, n))
    N = len(Q)
    if not (len(K) == len(n) == N):
        raise ValueError("Q, K and n must have the same length.")
    if not 1 <= N <= MAX_LAYERS:
        raise ValueError(f"Number of layers must be between 1 and {MAX_LAYERS}; got {N}.")
    if np.any(Q <= 0) or np.any(K <= 0) or np.any(n <= 0):
        raise ValueError("All Q, K and n must be positive.")
    Ri, Ro = float(R_wire), float(R_die)
    if not 0 < Ri < Ro:
        raise ValueError("Need 0 < R_wire < R_die.")
    V_in, V_out = float(V_wire), float(V_die)

    # ---- scales used to make the residuals dimensionless -----------------
    A = np.pi * (Ro ** 2 - Ri ** 2)                     # annulus area
    Q_tot = Q.sum()
    U = max(Q_tot / A, abs(V_in), abs(V_out))            # velocity scale
    frac0 = Q / Q_tot                                    # flow fraction of each layer
    h = Ro - Ri                                          # gap
    Kref = np.exp(np.sum(frac0 * np.log(K)))             # flow-weighted "average" fluid
    nref = np.sum(frac0 * n)
    gd_ref = (6.0 * Q_tot / A + abs(V_in - V_out)) / h   # typical shear rate
    tau_ref = Kref * gd_ref ** nref                      # typical stress

    # ---- map the unknown vector x to physical quantities -----------------
    # x[0], x[1] : wall stresses through asinh, so that both signs and many
    #              decades are covered smoothly (tau = tau_ref * sinh(x)).
    # x[2:]      : "softmax" weights for the layer thicknesses; they always give
    #              positive thicknesses that add up to the gap.
    def unpack(x):
        tau_i = tau_ref * np.sinh(x[0])
        tau_o = tau_ref * np.sinh(x[1])
        G, C = G_and_C_from_wall_stresses(tau_i, tau_o, Ri, Ro)
        w = np.concatenate([x[2:], [0.0]])
        t = np.exp(w - w.max()); t /= t.sum()
        radii = Ri + h * np.concatenate([[0.0], np.cumsum(t)]); radii[-1] = Ro
        return G, C, radii

    def pack(tau_i, tau_o, radii):                       # inverse of unpack
        t = np.maximum(np.diff(radii), 1e-300)
        w = np.log(t[:-1] / t[-1]) if N > 1 else np.array([])
        return np.concatenate([[np.arcsinh(tau_i / tau_ref), np.arcsinh(tau_o / tau_ref)], w])

    # ---- residuals: zero when all N + 1 equations are satisfied ----------
    def make_residuals(Kx, nx):
        def residuals(x):
            G, C, radii = unpack(x)
            with np.errstate(all="ignore"):
                u, Qc = march_through_layers(G, C, radii, Kx, nx, V_in)
                r = np.concatenate([(Qc - Q) / Q,               # N flow-rate errors (relative)
                                    [(u[-1] - V_out) / U]])     # die-wall velocity error
            return np.where(np.isfinite(r), r, 1e6)
        return residuals

    lb = np.concatenate([[-120.0, -120.0], -40 * np.ones(N - 1)])
    ub = -lb

    def run(x0, Kx, nx, nfev=400):
        """One nonlinear least-squares solve (trust-region reflective)."""
        x0 = np.clip(x0, lb + 1e-9, ub - 1e-9)
        return least_squares(make_residuals(Kx, nx), x0, bounds=(lb, ub), method="trf",
                             xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=nfev * (N + 2),
                             x_scale="jac")

    converged = 1e-20                                    # cost = 0.5 * sum(residual^2)

    # ---- Stage 1: exact uniform-fluid start, then a direct solve ---------
    try:
        x_ref = pack(*_uniform_fluid_solution(frac0, Q_tot, Kref, nref, Ri, Ro, V_in, V_out, tau_ref))
    except RuntimeError:
        x_ref = np.concatenate([[0.0, 0.0], np.log(frac0[:-1] / frac0[-1]) if N > 1 else []])
    best = run(x_ref, K, n)

    # ---- Stage 2: continuation (morph K and n from the reference fluid) --
    if best.cost > converged:
        x = run(x_ref, np.full(N, Kref), np.full(N, nref)).x
        lam, step = 0.0, 0.1
        while lam < 1.0 and step > 1e-4:
            lt = min(1.0, lam + step)
            Kx = np.exp((1 - lt) * np.log(Kref) + lt * np.log(K))
            nx = (1 - lt) * nref + lt * n
            trial = run(x, Kx, nx)
            if trial.cost < 1e-16:                       # step accepted: go on, slightly bigger
                x, lam, step = trial.x, lt, min(step * 1.5, 0.3)
            else:                                        # step failed: halve it
                step /= 2
        best = run(x, K, n)

    # ---- Stage 3: seeded multi-start (last resort) -----------------------
    if best.cost > converged:
        rng = np.random.default_rng(12345)
        for batch in range(4):
            screened = []
            for _ in range(100):
                if rng.random() < 0.5:
                    x0 = np.concatenate([rng.uniform(-5, 5, 2), rng.uniform(-3, 3, N - 1)])
                else:
                    x0 = x_ref + rng.normal(0, 1.0, N + 1)
                r = run(x0, K, n, nfev=25)               # cheap screening
                screened.append((r.cost, r.x))
            screened.sort(key=lambda t: t[0])
            for cost, x0 in screened[:6]:                # polish the 6 best
                trial = run(x0, K, n)
                if trial.cost < best.cost:
                    best = trial
                if best.cost < converged:
                    break
            if best.cost < converged:
                break
    if best.cost > 1e-15:
        raise RuntimeError(f"Solver did not converge (residual {np.sqrt(2*best.cost):.2e}).")

    # ---- package the result ---------------------------------------------
    G, C, radii = unpack(best.x)
    u, Qc = march_through_layers(G, C, radii, K, n, V_in)
    return dict(G=G, dpdz=-G, C=C, r0=zero_stress_radius(G, C),
                tau_wire=shear_stress(Ri, G, C), tau_die=shear_stress(Ro, G, C),
                radii=radii, u_edge=u, Q_calc=Qc, Q=Q, K=K, n=n,
                R_wire=Ri, R_die=Ro, V_wire=V_in, V_die=V_out,
                residual=np.sqrt(2 * best.cost))

### 4.8 Profiles for plotting

`profiles(sol)` evaluates the converged solution on a radial grid in each layer: velocity, stress, shear rate and viscosity. The viscosity is computed from Eq. 2.4 and is infinite at $r^*$ (see the question in Section 2.4).

In [ ]:
# ---------------------------------------------------------------------------
# Profiles for plotting: velocity, stress, shear rate and viscosity vs radius
# ---------------------------------------------------------------------------
def profiles(sol, pts_per_layer=200):
    """Evaluate the solution on a radial grid in each layer (for plots and post-processing).
    Returns one dict per layer with arrays r, u, tau, gdot (shear rate) and eta (viscosity).
    The velocity at each grid point is computed with the same exact quadrature as the solver."""
    out = []
    G, C = sol["G"], sol["C"]
    for i in range(len(sol["K"])):
        a, b = sol["radii"][i], sol["radii"][i + 1]
        K, n = sol["K"][i], sol["n"][i]
        r = np.linspace(a, b, pts_per_layer)
        u = np.array([sol["u_edge"][i] + (layer_integrals(a, rr, G, C, K, n)[0] if rr > a else 0.0)
                      for rr in r])
        tau = shear_stress(r, G, C)
        gdot = shear_rate(tau, K, n)
        with np.errstate(divide="ignore"):
            eta = K * np.abs(gdot) ** (n - 1)            # power-law viscosity (infinite where gdot = 0)
        out.append(dict(r=r, u=u, tau=tau, gdot=gdot, eta=eta))
    return out

### 4.9 Reporting and plotting

* `report(sol, names, L)` prints a summary in practical units, plus a table with one row per layer. The columns are:
  - `r_out`: the layer's outer radius in the die;
  - `t_die`: its thickness in the die;
  - `u_out`: the velocity at its outer edge;
  - `final t`: its final coating thickness after the die.
* `plot_profiles(sol, names)` draws four panels: the cross-section in the die, and the velocity, shear rate and viscosity against radius. The wire is at the bottom of each profile plot and the die at the top. A dashed line marks the interface, and a dotted line marks $r^*$ when it lies inside the gap.

In [ ]:
import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

LAYER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # one color per layer


def _regime(sol):
    """Classify the flow by the sign of G = -dp/dz (Section 2.9)."""
    G = sol["G"]
    scale = (abs(sol["tau_wire"]) + abs(sol["tau_die"])) / (sol["R_die"] - sol["R_wire"])
    if abs(G) < 1e-9 * scale:
        return "pure drag (no pressure gradient)"
    return ("pressure-assisted (pressure falls toward the die exit)" if G > 0
            else "pressure build-up (pressure rises toward the die exit)")


def report(sol, names=None, L=None):
    """Print the solution in practical units. L = die-land length [m] (optional)."""
    N = len(sol["K"])
    names = names or [f"Layer {i+1}" for i in range(N)]
    Rw, Rd, Vw = sol["R_wire"], sol["R_die"], sol["V_wire"]
    r = sol["radii"]
    pr = profiles(sol, 80)
    umin = min(p["u"].min() for p in pr); umax = max(p["u"].max() for p in pr)
    vscale = max(abs(Vw), abs(sol["V_die"]), sol["Q"].sum() / (np.pi * (Rd**2 - Rw**2)))
    if abs(umin) < 1e-9 * vscale: umin = 0.0                 # round-off at the stationary wall
    print(f"Wire radius {Rw*1e3:.4g} mm, die radius {Rd*1e3:.4g} mm, wire speed {Vw:.4g} m/s"
          + (f", die wall speed {sol['V_die']:.4g} m/s" if sol["V_die"] else ""))
    print(f"Pressure gradient  -dp/dz = {sol['G']:+.4e} Pa/m = {sol['G']*1e-6:+.4f} MPa/m   -> {_regime(sol)}")
    if L:
        print(f"Pressure change over land length {L*1e3:.3g} mm: p_in - p_out = {sol['G']*L*1e-6:+.3f} MPa")
    print(f"Wall shear stress: wire {sol['tau_wire']:+.4e} Pa, die {sol['tau_die']:+.4e} Pa")
    Fp = 2*np.pi*Rw*sol["tau_wire"]                          # Eq. 2.10, per metre of land
    print(f"Force of the melt on the wire: {Fp:+.4e} N per m of land" + (f"  ({Fp*L:+.4g} N over L)" if L else "")
          + ("  (resists the wire)" if Fp < 0 else ""))
    if sol["r0"] is not None and Rw < sol["r0"] < Rd:
        print(f"Zero-stress radius (velocity extremum) r* = {sol['r0']*1e3:.4f} mm")
    else:
        print("Zero-stress radius: outside the annulus (velocity monotonic from wire to die)")
    print(f"Velocity range {umin:.4g} to {umax:.4g} m/s" + ("   ** backflow present **" if umin < 0 else ""))
    print(f"Solver residual = {sol['residual']:.1e}\n")
    final = Vw > 0
    hdr = (f"{'':<12}{'Q [m3/s]':>11}{'K [Pa s^n]':>12}{'n':>7}{'Q frac':>8}{'r_out [mm]':>12}"
           f"{'t_die [mm]':>12}{'u_out [m/s]':>13}" + (f"{'final t [mm]':>14}" if final else ""))
    print(hdr); print("-" * len(hdr))
    if final:                                                # Eq. 2.9 solved for the final radii
        Rf = np.sqrt(Rw**2 + np.cumsum(sol["Q"]) / (np.pi * Vw))
        tf = np.diff(np.concatenate([[Rw], Rf]))
    for i in range(N):
        uo = sol["V_die"] if i == N - 1 else sol["u_edge"][i+1]
        print(f"{names[i]:<12}{sol['Q'][i]:>11.3e}{sol['K'][i]:>12.4g}{sol['n'][i]:>7.3f}"
              f"{sol['Q'][i]/sol['Q'].sum():>8.4f}{r[i+1]*1e3:>12.4f}{(r[i+1]-r[i])*1e3:>12.4f}"
              f"{uo:>13.4g}" + (f"{tf[i]*1e3:>14.4f}" if final else ""))
    if final:
        print(f"\nFinal coated radius {Rf[-1]*1e3:.4f} mm (die radius {Rd*1e3:.4g} mm); "
              "'final t' neglects swell and draw-down")


def _cross_section(ax, sol):
    """Concentric rings: wire (gray), layers (colors), die (dark)."""
    Rw, Rd, s = sol["R_wire"], sol["R_die"], 1e3
    ax.add_patch(Circle((0, 0), Rd*s*1.12, color="0.25"))
    for i in reversed(range(len(sol["K"]))):
        ax.add_patch(Circle((0, 0), sol["radii"][i+1]*s, color=LAYER_COLORS[i]))
    ax.add_patch(Circle((0, 0), Rw*s, color="#b0b0b0"))
    ax.set_xlim(-Rd*s*1.2, Rd*s*1.2); ax.set_ylim(-Rd*s*1.2, Rd*s*1.2)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Cross-section in the die", pad=14)
    ax.text(0, 0, "wire", ha="center", va="center", fontsize=8, color="0.15")


def plot_profiles(sol, names=None, title=None):
    """Cross-section, then velocity, shear-rate and viscosity profiles against radius."""
    N = len(sol["K"])
    names = names or [f"Layer {i+1}" for i in range(N)]
    pr = profiles(sol, 300)
    Rw, Rd = sol["R_wire"], sol["R_die"]
    fig = plt.figure(figsize=(15, 5.2))
    gs = fig.add_gridspec(1, 4, width_ratios=[0.9, 1.5, 1, 1])
    _cross_section(fig.add_subplot(gs[0]), sol)
    axes = [fig.add_subplot(gs[1])]
    axes += [fig.add_subplot(gs[k], sharey=axes[0]) for k in (2, 3)]
    for ax in axes:                                          # shading, interfaces, r*, grid
        for i in range(N):
            ax.axhspan(sol["radii"][i]*1e3, sol["radii"][i+1]*1e3, color=LAYER_COLORS[i], alpha=0.08, lw=0)
        for ri in sol["radii"][1:-1]:
            ax.axhline(ri*1e3, color="0.45", lw=0.8, ls="--")
        if sol["r0"] is not None and Rw < sol["r0"] < Rd:
            ax.axhline(sol["r0"]*1e3, color="0.2", lw=0.8, ls=":")
        _style(ax); ax.set_ylim(Rw*1e3, Rd*1e3)
    for ax in axes[1:]:
        plt.setp(ax.get_yticklabels(), visible=False)

    ax = axes[0]                                             # velocity
    for i, p in enumerate(pr):
        ax.plot(p["u"], p["r"]*1e3, color=LAYER_COLORS[i], lw=2.2, label=names[i])
    umin = min(0.0, min(p["u"].min() for p in pr)); umax = max(p["u"].max() for p in pr)
    pad = 0.05 * (umax - umin)
    ax.set_xlim(umin - (pad if umin < 0 else 0), umax + pad)
    ax.axvline(0, color="0.3", lw=0.8)
    ax.plot(ax.get_xlim(), [Rw*1e3]*2, color="#808080", lw=5, solid_capstyle="butt", clip_on=False)   # wire
    ax.plot(ax.get_xlim(), [Rd*1e3]*2, color="0.15", lw=4, solid_capstyle="butt", clip_on=False)      # die
    ax.set_xlabel("Axial velocity u [m/s]"); ax.set_ylabel("Radius r [mm]  (wire at bottom, die at top)")
    ax.set_title(f"Velocity  (wire {sol['V_wire']:.3g} m/s)", pad=14)
    ax.legend(loc="best", fontsize=9, frameon=False)

    ax = axes[1]                                             # shear rate
    for i, p in enumerate(pr):
        ax.plot(p["gdot"], p["r"]*1e3, color=LAYER_COLORS[i], lw=2)
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_xlabel("Shear rate du/dr [1/s]"); ax.set_title("Shear rate", pad=14)

    ax = axes[2]                                             # viscosity (log scale)
    for i, p in enumerate(pr):
        ok = np.isfinite(p["eta"])
        ax.plot(p["eta"][ok], p["r"][ok]*1e3, color=LAYER_COLORS[i], lw=2)
    if sol["r0"] is not None and Rw < sol["r0"] < Rd:        # power-law viscosity is infinite at r*: cap the axis
        far = np.concatenate([p["eta"][(np.abs(p["r"] - sol["r0"]) > 0.05*(Rd-Rw)) & np.isfinite(p["eta"])]
                              for p in pr])
        if far.size: ax.set_xlim(far.min()/1.5, far.max()*3)
    ax.set_xscale("log")
    lo, hi = ax.get_xlim()
    if np.log10(hi/lo) < 1.5:
        ax.xaxis.set_major_locator(matplotlib.ticker.LogLocator(base=10, subs=(1, 2, 5)))
        ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.set_xlabel(r"Viscosity $\eta$ [Pa s]"); ax.set_title("Local viscosity", pad=14)

    fig.suptitle(title or f"{N}-layer annular power-law flow (-dp/dz = {sol['G']*1e-6:+.3g} MPa/m; "
                 "dashed = interface, dotted = zero stress)", fontsize=12)
    fig.tight_layout(); plt.show()
    return fig


def _style(ax):
    """Light grid, no top/right frame lines."""
    ax.grid(alpha=0.25, lw=0.6)
    for side in ("top", "right"): ax.spines[side].set_visible(False)


def plain_log_axis(axis):
    """Label a log axis with plain numbers (1, 2, 5, 10, ...) instead of powers of ten."""
    axis.set_major_locator(matplotlib.ticker.LogLocator(base=10, subs=(1, 2, 5)))
    axis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_formatter(matplotlib.ticker.NullFormatter())

### 4.10 A first call: anatomy of a solution

Before verifying the solver, here is a minimal example of how to call it and what comes back. The case is two layers in the gap between a 0.5 mm-radius wire and a 1.0 mm-radius die, with the wire moving at 1 m/s.

In [ ]:
sol_demo = solve_annular(Q=[3.0e-7, 9.0e-7],        # flow rates of inner and outer layer [m^3/s]
                         K=[4000.0, 12000.0],        # consistencies [Pa s^n]
                         n=[0.40, 0.30],             # power-law indices [-]
                         R_wire=0.5e-3, R_die=1.0e-3,
                         V_wire=1.0)                 # line speed [m/s]

print("-dp/dz             =", f"{sol_demo['G']:.4e} Pa/m")
print("interface radius   =", f"{sol_demo['radii'][1]*1e3:.4f} mm")
print("interface velocity =", f"{sol_demo['u_edge'][1]:.4f} m/s")
print("wall stresses      =", f"wire {sol_demo['tau_wire']:.4e} Pa, die {sol_demo['tau_die']:.4e} Pa")
print("residual           =", f"{sol_demo['residual']:.1e}")

<a id="part5"></a>
# 5. Verification against exact solutions

Each test below targets a different part of the code. All differences should be at the level of **round-off**, about $10^{-13}$ or smaller. A larger error would point to a bug.

### Test 1 — Newtonian drag flow (BSL Problem 2B.7)

**Tests:** quadrature, marching, the solver and the sign conventions, in the simplest case. We set $n = 1$ and feed in exactly the flow rate that drag alone produces (Eq. 2.13). The solver must then find **$G = 0$** and the exact wire stress.

In [ ]:
from scipy.integrate import quad

Rw, Rd = 0.5e-3, 1.0e-3
kap = Rw / Rd
mu, V = 500.0, 0.5
Q_bsl = np.pi*Rd**2*V/2*((1 - kap**2)/np.log(1/kap) - 2*kap**2)          # Eq. 2.13
s1 = solve_annular([Q_bsl], [mu], [1.0], Rw, Rd, V_wire=V)
tau_exact = -mu*V/(Rw*np.log(Rd/Rw))
print(f"-dp/dz found by the solver: {s1['G']:.1e} Pa/m   (exact: 0; compare with stress/gap ~ {abs(tau_exact)/(Rd-Rw):.0e} Pa/m)")
print(f"wire stress: solver {s1['tau_wire']:.10e} Pa, exact {tau_exact:.10e} Pa, rel. error {s1['tau_wire']/tau_exact - 1:+.1e}")

### Test 2 — Newtonian Couette–Poiseuille flow

**Tests:** the pressure term, for both signs of $G$ and both directions of wire motion. The exact Newtonian profile (Eq. 2.12) is linear in $G$, so the flow rate is $Q(G) = Q(0) + G\,\partial Q/\partial G$. The exact $G$ for any required $Q$ follows directly, with the integrals evaluated by SciPy's adaptive `quad`.

In [ ]:
Lg = np.log(Rd/Rw)
def Q_newtonian(G, V):
    u = lambda r: G/(4*mu)*((Rd**2 - r**2) - (Rd**2 - Rw**2)*np.log(Rd/r)/Lg) + V*np.log(Rd/r)/Lg   # Eq. 2.12
    return quad(lambda r: 2*np.pi*u(r)*r, Rw, Rd, epsabs=0, epsrel=1e-13)[0]

for V, Q in [(0.0, 1e-7), (0.5, 2e-7), (2.0, 1e-7), (-0.3, 2e-7)]:
    G_exact = (Q - Q_newtonian(0, V)) / Q_newtonian(1, 0)
    sn = solve_annular([Q], [mu], [1.0], Rw, Rd, V_wire=V)
    print(f"V_w = {V:+.1f} m/s, Q = {Q:.0e} m^3/s:  -dp/dz solver {sn['G']:+.8e}, exact {G_exact:+.8e},"
          f" rel. error {sn['G']/G_exact - 1:+.0e}")

**What to notice:** $G$ changes sign. A fast wire with a modest flow rate needs a *negative* $G$, so the pressure rises toward the exit, as in Section 2.9.

### Test 3 — Power-law pure drag

**Tests:** the power-law part of the code (`shear_rate` with $n \ne 1$). Equation 2.14 gives the exact drag-flow velocity. We integrate it to get the drag flow rate and give that to the solver, which must return $G = 0$ and the exact constant $C$.

In [ ]:
K, n = 5000.0, 0.4; s = 1/n; V = 1.0
I = (Rd**(1-s) - Rw**(1-s))/(1-s)
C_abs = K*(V/I)**n                                   # from u(R_d) = 0 in Eq. 2.14
u_drag = lambda r: V - (C_abs/K)**s*(r**(1-s) - Rw**(1-s))/(1-s)
Q_drag = quad(lambda r: 2*np.pi*u_drag(r)*r, Rw, Rd, epsabs=0, epsrel=1e-13)[0]
sd = solve_annular([Q_drag], [K], [n], Rw, Rd, V_wire=V)
print(f"-dp/dz = {sd['G']:.1e} Pa/m (exact 0);  C/C_exact - 1 = {sd['C']/(-C_abs) - 1:+.1e}")

### Test 4 — One fluid split into two "layers"

**Tests:** the multilayer machinery: interface placement, velocity continuity and the per-layer flow rates. If both layers are the same fluid, the interface is fictitious, and the pressure gradient must equal the single-layer result.

In [ ]:
one = solve_annular([3e-7], [K], [n], Rw, Rd, V_wire=1.0)
two = solve_annular([1e-7, 2e-7], [K, K], [n, n], Rw, Rd, V_wire=1.0)
print(f"single layer -dp/dz = {one['G']:.10e},  split into two = {two['G']:.10e},  rel. difference {two['G']/one['G'] - 1:+.1e}")

### Test 5 — Independent re-integration of a hard case

**Tests:** the special quadrature near the zero-stress radius (Section 3.2). We choose a thin coating at high speed, which gives an adverse pressure gradient, backflow, and $r^*$ **inside** the outer layer. From the converged $G$, $C$ and interface radius, the flow rates and the die-wall velocity are recomputed with SciPy's adaptive `quad`, told explicitly where $r^*$ is. They must reproduce the required flow rates and $u(R_d) = 0$.

In [ ]:
Qs, Ks, ns = [3e-8, 6e-8], [4000., 12000.], [0.4, 0.3]
sv = solve_annular(Qs, Ks, ns, Rw, Rd, V_wire=1.5)
G, C, rr = sv["G"], sv["C"], sv["radii"]
print(f"r* = {sv['r0']*1e3:.4f} mm lies inside the annulus ({Rw*1e3} to {Rd*1e3} mm); "
      f"min velocity {min(p['u'].min() for p in profiles(sv)):+.3f} m/s (backflow)")
u_a = 1.5
for i in range(2):
    g = lambda r: np.sign(-G*r/2 + C/r)*abs((-G*r/2 + C/r)/Ks[i])**(1/ns[i])     # Eqs. 2.3 and 2.5
    pts = [sv["r0"]] if (rr[i] < sv["r0"] < rr[i+1]) else None
    a, b = rr[i], rr[i+1]
    u_b = u_a + quad(g, a, b, points=pts, epsabs=0, epsrel=1e-12, limit=300)[0]
    Q_i = 2*np.pi*((b*b - a*a)/2*u_a + quad(lambda r: g(r)*(b*b - r*r)/2, a, b, points=pts,
                                            epsabs=0, epsrel=1e-12, limit=300)[0])
    print(f"layer {i+1}: Q from independent quadrature / required Q - 1 = {Q_i/Qs[i] - 1:+.1e}")
    u_a = u_b
print(f"die-wall velocity from independent quadrature: {u_a:+.1e} m/s (must be 0)")

<a id="part6"></a>
# 6. Worked examples

### Example 1 — A two-layer insulated wire

**Problem.** A 1.0 mm-diameter copper wire ($R_w = 0.5$ mm) runs at **1 m/s** through a die land of radius $R_d = 1.0$ mm and length $L = 5$ mm. It is to receive **0.10 mm of insulation** under **0.20 mm of jacket**, so the final coated radius is 0.80 mm. Find the die pressure drop, the force on the wire, and where the interface sits in the die.

**Materials** (illustrative values, typical of polyolefin and PVC-type melts at processing temperature):

| layer | $K$ [Pa·sⁿ] | $n$ |
|---|---|---|
| Insulation (inner) | 4000 | 0.40 |
| Jacket (outer) | 12000 | 0.30 |

**Solution steps:** (1) convert the thicknesses to flow rates with Eq. 2.9; (2) solve; (3) report and plot.

In [ ]:
Rw, Rd = 0.50e-3, 1.00e-3      # wire and die-land radius [m]
Vw     = 1.0                   # line speed [m/s]
L      = 5e-3                  # land length [m]
names  = ["Insulation", "Jacket"]
K      = [4000.0, 12000.0]     # Pa s^n
n      = [0.40,   0.30]
t_final = [0.10e-3, 0.20e-3]   # final coating thicknesses [m]

Q = flow_rates_from_coating(Rw, t_final, Vw)      # step 1: Eq. 2.9
print("flow rates from Eq. 2.9:", np.array2string(Q, precision=4), "m^3/s\n")
sol1 = solve_annular(Q, K, n, Rw, Rd, V_wire=Vw)  # step 2
report(sol1, names, L=L)                          # step 3
plot_profiles(sol1, names, title="Example 1: two-layer wire coating in the die land");

**How to read the output**

* **Pressure gradient** $-dp/dz > 0$, so the die must *push*. The coating (0.80 mm final radius) is thicker than the wire alone can drag through a 1.0 mm die (see Example 3). Multiplying by $L$ gives the pressure drop across the land, about 1.4 MPa (14 bar).
* **Wall stresses** are both negative. At the wire, the melt next to the wire moves slower than the wire and resists it. At the die, the velocity falls toward the stationary wall ($du/dr < 0$).
* **Force on the wire**, about −1.2 N: the land alone resists the wire with roughly 1.2 N of force.
* **Layer table.** The insulation is **0.111 mm** thick in the die but 0.100 mm on the finished wire. In the die it moves *slower* than the wire on average (between 1.0 and about 0.78 m/s), so it needs more cross-section to carry the same flow. After the die it speeds up to the wire speed and thins. The jacket behaves the same way, more strongly.

**What to notice in the plots**

* The **velocity** falls monotonically from the wire speed to zero at the die, and $r^*$ lies outside the gap. The profile has a **kink** at the interface.
* The **shear rate** jumps at the interface. Stress is continuous there (Eq. 2.6b), but $\dot\gamma = (|\tau|/K)^{1/n}$ differs between the two materials.
* The shear rates reach **thousands of s⁻¹**, and the viscosities are only tens of Pa·s, far below the zero-shear viscosities of these melts. This is typical of wire coating, and it is why the power law, a high-shear-rate model, is reasonable here.

### Revisiting assumption A2: viscous heating

All the mechanical energy put into the melt is dissipated as heat: pressure work $Q\,\Delta p$ plus the work done by the wire, $|F|\,V_w$. If none of it were conducted away (the **adiabatic** worst case), the average temperature rise of the melt would be
$$\Delta T_\text{ad} = \frac{Q_\text{tot}\,\Delta p + |F|\,V_w}{\rho\,c_p\,Q_\text{tot}} .$$

In [ ]:
rho, cp = 900.0, 2300.0                          # melt density [kg/m^3] and heat capacity [J/(kg K)]
Q_tot = sol1["Q"].sum()
dp    = sol1["G"] * L                            # pressure drop over the land [Pa]
F     = abs(2*np.pi*Rw*sol1["tau_wire"]*L)       # wire force over the land [N]
P_mech = Q_tot*dp + F*Vw                          # mechanical power dissipated in the melt [W]
dT_ad = P_mech / (rho*cp*Q_tot)
print(f"pressure work {Q_tot*dp:.3f} W + wire work {F*Vw:.3f} W = {P_mech:.3f} W dissipated")
print(f"adiabatic temperature rise of the melt in the land: {dT_ad:.1f} K")

A temperature rise of about 1 K is negligible compared with typical melt-temperature tolerances, so the isothermal assumption is acceptable here. Both terms grow with line speed and with land length (Exercise C6). The dissipation is also concentrated near the wire, where the shear rate is highest, so the local temperature rise there exceeds this average. **Long lands and high line speeds call for a thermal analysis.**

### Example 2 — Effect of line speed (fixed coating)

Increasing the line speed at a fixed coating thickness raises both flow rates in proportion to $V_w$ (Eq. 2.9). For a **single** power-law fluid this scales the whole velocity field. Stresses then scale exactly as $V_w^{\,n}$, and the interface would not move.

With **two different** $n$ values the scaling is no longer exact. The cell below sweeps the line speed from 0.1 to 10 m/s and fits the exponents.

In [ ]:
speeds = np.geomspace(0.1, 10, 21)
res = [solve_annular(flow_rates_from_coating(Rw, t_final, V), K, n, Rw, Rd, V_wire=V) for V in speeds]
G_v  = np.array([s["G"] for s in res])
ri_v = np.array([s["radii"][1] for s in res])
F_v  = np.array([-2*np.pi*Rw*s["tau_wire"]*L for s in res])

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].plot(speeds, G_v*1e-6, color=LAYER_COLORS[0], lw=2); axs[0].set_xscale("log")
axs[0].set_ylabel("-dp/dz [MPa/m]"); axs[0].set_title("Pressure gradient in the land")
axs[1].plot(speeds, ri_v*1e3, color=LAYER_COLORS[1], lw=2); axs[1].set_xscale("log")
axs[1].set_ylabel("Interface radius in die [mm]"); axs[1].set_title("Interface position")
axs[2].loglog(speeds, F_v, color=LAYER_COLORS[2], lw=2); plain_log_axis(axs[2].yaxis)
axs[2].set_ylabel(f"Wire pull force over L = {L*1e3:.0f} mm [N]"); axs[2].set_title("Drag force on the wire")
for ax in axs:
    ax.set_xlabel("Line speed [m/s]"); _style(ax)
fig.tight_layout(); plt.show()

kF = np.polyfit(np.log(speeds), np.log(F_v), 1)[0]
kG = np.polyfit(np.log(speeds), np.log(np.abs(G_v)), 1)[0]
print(f"interface radius moves from {ri_v[0]*1e3:.4f} mm to {ri_v[-1]*1e3:.4f} mm over 0.1-10 m/s")
print(f"fitted exponents:  -dp/dz ~ V^{kG:.2f},  wire force ~ V^{kF:.2f}   (n = {n[0]} and {n[1]})")

**What to notice**

* The **wire force** scales as $V_w^{\,0.40}$, the inner layer's index. Only the inner layer touches the wire, and $\tau_w \propto \dot\gamma^{\,n_1}$.
* The **pressure gradient** scales roughly as $V_w^{0.25}$, *below both* $n$ values. That is not a contradiction: $G$ is the **difference** between what the die must supply and what drag provides, and a difference need not follow either component's scaling. With equal indices (Exercise 3) it would be exactly $V_w^{\,n}$.
* The **interface** moves by only a few micrometres over two decades of speed. Its position in the die is set mainly by the split between the two coatings.
* Doubling the line speed **does not double** the force or the pressure: shear thinning helps high-speed coating.

### Example 3 — Coating thickness and the sign of the pressure gradient

Now fix the line speed (1 m/s) and the inner:outer thickness ratio (1:2), and vary the **total** coating thickness. A thin coating means the wire drags out more melt than is delivered, so the die must hold it back. A thick coating means the die must push.

In [ ]:
t_tot = np.linspace(0.04e-3, 0.48e-3, 23)
sw = [solve_annular(flow_rates_from_coating(Rw, [t/3, 2*t/3], Vw), K, n, Rw, Rd, V_wire=Vw) for t in t_tot]
G_t = np.array([s["G"] for s in sw])
umin_t = np.array([min(p["u"].min() for p in profiles(s, 60)) for s in sw])
i0 = np.where(np.diff(np.sign(G_t)))[0][0]                       # linear interpolation for G = 0
t0 = t_tot[i0] - G_t[i0]*(t_tot[i0+1] - t_tot[i0])/(G_t[i0+1] - G_t[i0])
print(f"drag-only total coating thickness (dp/dz = 0): about {t0*1e3:.3f} mm at {Vw} m/s")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(t_tot*1e3, G_t*1e-6, color=LAYER_COLORS[0], lw=2)
axs[0].axhline(0, color="0.3", lw=0.8); axs[0].axvline(t0*1e3, color="0.5", lw=0.8, ls="--")
axs[0].set_ylabel("-dp/dz [MPa/m]"); axs[0].set_title("Pressure gradient vs final coating thickness")
axs[1].plot(t_tot*1e3, np.minimum(umin_t, 0), color=LAYER_COLORS[1], lw=2)
axs[1].axhline(0, color="0.3", lw=0.8)
axs[1].set_ylabel("Most negative velocity [m/s]"); axs[1].set_title("Backflow appears only for very thin coatings")
for ax in axs:
    ax.set_xlabel("Total final coating thickness [mm]"); _style(ax)
fig.tight_layout(); plt.show()

thin = sw[0]
report(thin, names, L=L)
plot_profiles(thin, names, title=f"Example 3: thin coating ({t_tot[0]*1e3:.2f} mm total): pressure build-up and backflow");

**What to notice**

* The pressure gradient **changes sign** at the "drag-only" thickness, about 0.14 mm at 1 m/s in this die. Thinner coatings need pressure to **rise** toward the exit ($-dp/dz < 0$), and thicker ones need the die to push.
* For the thinnest coating, the zero-stress radius $r^*$ (dotted line) lies **inside** the jacket. The velocity has a minimum there, and the melt near the die wall flows slightly **backward**. In the viscosity panel the power-law viscosity becomes very large near $r^*$, the artefact discussed in Section 2.4.
* Most of the jacket in the thin case is nearly **stagnant**. Its shear is concentrated near the wire because the jacket is strongly shear-thinning. In practice such stagnant regions are undesirable, because the melt stays in the die a long time and can degrade.
* The curve bends where $r^*$ passes through the die wall, which is the transition between monotonic profiles and profiles with backflow.

<a id="part7"></a>
# 7. Your own calculation ✏️

Edit the inputs in the next cell and run it. List the layers **from the wire outward**. You can use 1 to 5 layers; a standard insulated wire has 2.

**Typical parameter ranges** (order-of-magnitude guide for wire coating):

| input | typical range | note |
|---|---|---|
| `R_wire` | 0.1 – 2 mm | from fine magnet wire to building wire |
| `R_die − R_wire` (gap) | 0.1 – 1 mm | |
| `V_wire` | 0.1 – 20 m/s | higher speeds need care with viscous heating |
| `K` | 10³ – 5×10⁴ Pa·sⁿ | strongly temperature dependent |
| `n` | 0.2 – 0.6 | 1 for a Newtonian fluid |
| final coating thickness | 0.02 – 1 mm per layer | |

**Choose how to specify the flow:** set `use_thickness = True` and give the final thicknesses `t_final` (Eq. 2.9 then sets the flow rates), or set it to `False` and give the flow rates `Q` directly.

**Troubleshooting**

| message or symptom | likely cause and fix |
|---|---|
| `ValueError: ...` | An input is outside its allowed range (for example $R_d \le R_w$, or a negative flow rate). Read the message. |
| `RuntimeError: Solver did not converge` | Extreme inputs, such as a viscosity ratio of 10⁴ or more combined with a very thin layer. Check the units, then move the inputs toward the typical ranges. |
| The run is slow (more than 10 s) | The solver is using continuation for strongly mismatched layers. This is normal for extreme cases. |
| "backflow present" | A thin coating in a wide die (Example 3). This is physical, but often undesirable in practice. |
| Very large pressure drop | The coating is thick relative to the gap, or the line speed is high. Compare with Example 3. |

In [ ]:
# ---------------- INPUTS (edit me) ----------------
R_wire = 0.40e-3          # wire radius [m]
R_die  = 0.85e-3          # die-land radius [m]
V_wire = 2.0              # line speed [m/s]
V_die  = 0.0              # die wall speed [m/s] (normally 0)
L      = 4e-3             # land length [m]

names = ["Inner", "Outer"]        # labels, wire outward
K     = [3000.0, 9000.0]          # consistency [Pa s^n]
n     = [0.45,   0.32]            # power-law index [-]

use_thickness = True
t_final = [0.08e-3, 0.22e-3]      # final coating thicknesses [m]   (used if use_thickness = True)
Q       = [3.0e-7, 1.2e-6]        # flow rates [m^3/s]              (used if use_thickness = False)
# ---------------------------------------------------

if use_thickness:
    Q = flow_rates_from_coating(R_wire, t_final, V_wire)
sol = solve_annular(Q, K, n, R_wire, R_die, V_wire=V_wire, V_die=V_die)
report(sol, names, L=L)
plot_profiles(sol, names);

<a id="part8"></a>
# 8. Exercises

The exercises go from analytical (A) through computational (C) to open-ended design questions (D). Most computational ones need only a few lines that call `solve_annular`, `flow_rates_from_coating` and `profiles`. Add new cells below this one (*Insert → Insert Cell Below*).

**A1 (derivation).** Starting from Eq. 2.3 with $n = 1$, derive the Newtonian velocity profile, Eq. 2.12. Then set $G = 0$ and integrate to recover the BSL drag-flow rate, Eq. 2.13.

**A2 (power-law drag flow).** Derive Eq. 2.14 for a single power-law fluid in pure drag flow. What happens to it as $n \to 1$? *Hint:* $\lim_{s\to1}(r^{1-s} - R_w^{1-s})/(1-s) = \ln(r/R_w)$.

**A3 (thin-gap limit).** Let $h = R_d - R_w \ll R_w$. Show that $\tau(r)$ in Eq. 2.3 becomes approximately linear in $r - R_w$, as in a slit. Estimate the relative size of the curvature correction in terms of $h/R_w$.

**C1 (scaling check).** Repeat the line-speed sweep of Example 2 with **equal** indices, $n = [0.35, 0.35]$. Fit the exponent of $-dp/dz$ against $V_w$ and confirm that it equals $n$. Explain why the interface no longer moves.

**C2 (material order).** Swap the two materials in Example 1 (stiff inner layer, soft outer layer), keeping the same final thicknesses. How do the pressure drop, the wire force and the in-die interface radius change? Explain the change in wire force using the viscosity at the wire.

**C3 (die design).** For the Example 1 coating (0.10 + 0.20 mm at 1 m/s), find the die radius $R_d$ for which the land needs **zero** pressure gradient. *Hint:* write a function `f(Rd)` that returns `solve_annular(...)["G"]`, check its sign at a few radii, and use `scipy.optimize.brentq` on a bracket where it changes sign. A suitable bracket is 1.5–5 mm. Is the resulting die practical? What does this tell you about why wire-coating dies of this kind run with positive die pressure?

**C4 (surface defects).** Many melts develop surface defects ("sharkskin") when the wall shear stress at the die exit exceeds roughly 0.1–0.2 MPa; the critical value depends on the polymer. For Example 1, find the line speed at which $|\tau_d|$ reaches 0.15 MPa. Which constraint would limit the line speed first: pressure, wire force or surface quality?

**C5 (numerics).** In Section 4.3, reduce the number of quadrature nodes (for example `N_LEGENDRE = 6` and `N_JACOBI = 4`), re-run that cell, then re-run Tests 1–5 and Example 1. Which results change, and by how much? For each test, work out which quadrature rule was used: compare the zero-stress radius `sol["r0"]` with the layer limits, as in `layer_integrals`. Explain your observations using the factorization in Eq. 3.1. Restore the original values (48 and 40) and re-run the cell afterwards.

**C6 (viscous heating).** Using the adiabatic estimate after Example 1, compute $\Delta T_\text{ad}$ at 20 m/s for the 5 mm land and for a 20 mm land. How does $\Delta T_\text{ad}$ scale with line speed and with land length? Which term contributes more, pressure work or wire work? Why is the local temperature rise near the wire larger than this average?

**D1 (three layers).** Add a thin tie layer, 0.02 mm final thickness with $K = 2000$ Pa·sⁿ and $n = 0.5$, between the insulation and the jacket. How thick is it in the die? What happens to the shear-rate jumps at its two interfaces?

**D2 (model limits).** Identify two operating conditions from Examples 1–3 where the power-law model is least trustworthy, and explain why. The companion notebook with Carreau viscosity and wall slip is a way to test your answer.

<a id="part9"></a>
# 9. Summary and references

**Key ideas**

1. In fully developed annular flow, the shear stress is $\tau = -Gr/2 + C/r$ **in every layer**. The pressure gradient is uniform across the gap, and $r\tau$ is continuous at interfaces.
2. The constitutive equation turns the known stress into a known shear rate. Integrating outward from the wire gives the velocity, and a single integral (after integration by parts) gives each layer's flow rate.
3. The multilayer problem has $N + 1$ unknowns: two stress constants and $N - 1$ interfaces. It has $N + 1$ equations: $N$ flow rates and no slip at the die. They are solved with nonlinear least squares from a carefully constructed starting point.
4. In wire coating, the **final coating thicknesses fix the flow rates** (Eq. 2.9), and the sign of the pressure gradient depends on whether the coating is thicker or thinner than the wire can drag through the die by itself.
5. Shear thinning makes pressure and wire force grow much more slowly than linearly with line speed. The interface position is set mainly by the coating split.
6. A model is only as good as its **verification** (exact limits, internal consistency, independent re-integration) and its **assumptions** (creeping flow, isothermal, power law, no slip, stable interfaces). Check them for every new operating point.

**References**

* R. B. Bird, W. E. Stewart, E. N. Lightfoot, *Transport Phenomena*, 2nd ed., Wiley (2002): shell balances (Chapter 2); Problem 2B.7, annular flow with the inner cylinder moving axially.
* J.-F. Agassant, P. Avenas, P. J. Carreau, B. Vergnes, M. Vincent, *Polymer Processing: Principles and Modeling*, 2nd ed., Hanser (2017): die flows, rheology of melts, coextrusion.
* C. D. Han, D. Rao, "The rheology of wire coating extrusion," *Polym. Eng. Sci.* 18, 1019–1029 (1978).
* J. F. Carley, T. Endo, W. Krantz, "Realistic analysis of flow in wire coating dies," *Polym. Eng. Sci.* 19, 1178–1187 (1979).
* H. H. Winter, SPE ANTEC Technical Papers 23, 462 (1977), on wire coating (as cited by Z. Khan et al., *Coatings* 9, 147 (2019)).
* Z. Tadmor, C. G. Gogos, *Principles of Polymer Processing*, 2nd ed., Wiley (2006): general background on extrusion and die flows.